In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

if "df" not in globals():
    df = pd.read_csv("../Online Retail.csv")

df_raw = df.copy()

print("Original dataset shape:", df_raw.shape)
print(df_raw.head())

Original dataset shape: (541909, 8)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

    InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/10 8:26       2.55     17850.0  United Kingdom  
1  12/1/10 8:26       3.39     17850.0  United Kingdom  
2  12/1/10 8:26       2.75     17850.0  United Kingdom  
3  12/1/10 8:26       3.39     17850.0  United Kingdom  
4  12/1/10 8:26       3.39     17850.0  United Kingdom  


## سؤال ۱: آیا نوع داده‌ها و قالب تاریخ برای آماده‌سازی مناسب است؟

### پاسخ

در این مرحله باید نوع ستون‌ها بررسی شود و ستون `InvoiceDate` به قالب استاندارد تاریخ و زمان تبدیل شود. ستون‌های شناسه مانند `InvoiceNo`، `StockCode` و `CustomerID` نیز باید به‌عنوان شناسه تفسیر شوند، نه مقادیر عددی برای محاسبات ریاضی.

### چرا این سؤال را می‌پرسیم؟

محاسبه فاصله زمانی خریدها، ساخت ویژگی‌های مبتنی بر زمان و تعیین بازه مشاهده و برچسب ریزش به تاریخ معتبر نیاز دارد. نوع نامناسب ستون‌ها می‌تواند باعث خطا یا محاسبات نادرست شود.

### تصمیم آماده‌سازی

ابتدا نوع داده‌ها را بررسی می‌کنیم و تاریخ‌ها را به قالب `datetime` تبدیل می‌کنیم. تعداد تاریخ‌های غیرقابل‌تبدیل نیز گزارش می‌شود تا مشکلات احتمالی پنهان نمانند.

In [53]:
df_prep = df_raw.copy()

print("Data types before preparation:")
print(df_prep.dtypes)

df_prep["InvoiceDate"] = pd.to_datetime(
    df_prep["InvoiceDate"],
    format="%m/%d/%y %H:%M",
    errors="coerce"
)

print("\nInvalid dates:", df_prep["InvoiceDate"].isna().sum())
print("Date range:")
print(df_prep["InvoiceDate"].min())
print(df_prep["InvoiceDate"].max())

print("\nData types after preparation:")
print(df_prep.dtypes)

Data types before preparation:
InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object

Invalid dates: 0
Date range:
2010-12-01 08:26:00
2011-12-09 12:50:00

Data types after preparation:
InvoiceNo                 str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object


In [54]:
duplicate_counts = (
    df.groupby(list(df.columns), dropna=False)
      .size()
      .reset_index(name="Count")
)

duplicate_counts = duplicate_counts[
    duplicate_counts["Count"] > 1
]

duplicate_counts = duplicate_counts.drop_duplicates().copy()

display(duplicate_counts)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Count
491,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,12/1/10 11:45,1.25,17908.0,United Kingdom,2
499,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,12/1/10 11:45,4.95,17908.0,United Kingdom,2
510,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,12/1/10 11:45,2.10,17908.0,United Kingdom,2
511,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,12/1/10 11:45,2.95,17908.0,United Kingdom,2
537,536412,21448,12 DAISY PEGS IN WOOD BOX,1,12/1/10 11:49,1.65,17920.0,United Kingdom,2
...,...,...,...,...,...,...,...,...,...
534958,C572226,85066,CREAM SWEETHEART MINI CHEST,-1,10/21/11 13:58,12.75,15321.0,United Kingdom,2
535288,C574095,22326,ROUND SNACK BOXES SET OF4 WOODLAND,-1,11/3/11 9:54,2.95,12674.0,France,2
535377,C574510,22360,GLASS JAR ENGLISH CONFECTIONERY,-1,11/4/11 13:25,2.95,15110.0,United Kingdom,2
535661,C575940,23309,SET OF 60 I LOVE LONDON CAKE CASES,-24,11/13/11 11:38,0.55,17838.0,United Kingdom,2


## سؤال ۲: با داده‌های گمشده و ردیف‌های تکراری چگونه برخورد کنیم؟

### پاسخ
در مرحله Data Understanding مشخص شد که برخی ردیف‌ها `CustomerID` یا `Description` ندارند و تعدادی ردیف کاملاً تکراری نیز وجود دارد.

برای پروژه پیش‌بینی ریزش، وجود شناسه مشتری ضروری است؛ زیرا بدون آن نمی‌توان سابقه خرید را به مشتری مشخصی نسبت داد.

ردیف‌های کاملاً تکراری نیز باید بررسی شوند. در این پروژه، حذف ردیف‌های تکراری کامل به‌عنوان یک تصمیم آماده‌سازی در نظر گرفته می‌شود؛ اما نسخه خام داده حفظ خواهد شد.

مقادیر گمشده `CustomerID` را برای تحلیل مشتری‌محور کنار می‌گذاریم، نه اینکه برای آن‌ها شناسه ساختگی تولید کنیم. مقادیر گمشده `Description` به‌تنهایی مانع محاسبه ویژگی‌های اصلی مشتری نیستند.

### چرا این سؤال را می‌پرسیم؟

داده‌های تکراری می‌توانند تعداد تراکنش‌ها و مقدار خرید را بیش از واقع نشان دهند. نبود شناسه مشتری نیز مانع ساخت ویژگی‌ها و برچسب قابل اتکا برای همان مشتری می‌شود.

### تصمیم آماده‌سازی

ابتدا تعداد موارد را ثبت می‌کنیم، سپس یک نسخه آماده‌سازی‌شده می‌سازیم. داده خام را تغییر نمی‌دهیم و هیچ شناسه مشتری‌ای را حدس نمی‌زنیم.

In [55]:
print("Missing values before preparation:")
print(df_prep.isna().sum())

print("\nExact duplicate rows:")
print(df_prep.duplicated().sum())

rows_before = len(df_prep)

df_prep = df_prep.drop_duplicates().copy()

print("\nRows removed as exact duplicates:", rows_before - len(df_prep))
print("Rows remaining:", len(df_prep))

missing_customer_rows = df_prep["CustomerID"].isna().sum()
print("\nRows with missing CustomerID:", missing_customer_rows)

df_prep = df_prep.dropna(subset=["CustomerID"]).copy()

df_prep["CustomerID"] = df_prep["CustomerID"].astype("int64")

print("Rows after excluding missing CustomerID:", len(df_prep))
print("Unique customers:", df_prep["CustomerID"].nunique())

Missing values before preparation:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Exact duplicate rows:
5268

Rows removed as exact duplicates: 5268
Rows remaining: 536641

Rows with missing CustomerID: 135037
Rows after excluding missing CustomerID: 401604
Unique customers: 4372


## سؤال ۳: کدام تراکنش‌ها برای تحلیل رفتار خرید معتبرند؟

### پاسخ
در دیتاست، همه ردیف‌ها لزوماً خرید عادی نیستند. فاکتورهایی که شماره آن‌ها با C شروع می‌شود، با لغو فاکتور مرتبط‌اند. همچنین مقدار منفی Quantity معمولاً با مرجوعی یا اصلاح تراکنش ارتباط دارد و قیمت صفر یا منفی نیازمند بررسی است.

برای ساخت ویژگی‌های رفتار خرید، یک مجموعه جداگانه از تراکنش‌های خرید معتبر می‌سازیم که در آن:

- شماره فاکتور با C شروع نشده باشد.
- Quantity مثبت باشد.
- UnitPrice مثبت باشد.
- CustomerID موجود باشد.
- InvoiceDate معتبر باشد.

### چرا این سؤال را می‌پرسیم؟

اگر مرجوعی‌ها و لغوها را مانند خرید عادی در نظر بگیریم، تعداد خریدها، مبلغ خرید و فاصله بین خریدها ممکن است نادرست محاسبه شوند.

### تصمیم آماده‌سازی

برای تحلیل رفتار خرید، فقط ردیف‌های منطبق با تعریف بالا را وارد مجموعه خرید معتبر می‌کنیم. این تعریف یک تصمیم عملیاتی برای پروژه است؛ به این معنا نیست که تمام تراکنش‌های حذف‌شده خطا هستند. نسخه خام برای بررسی جداگانه آن‌ها باقی می‌ماند.

In [56]:
cancelled_mask = (
    df_prep["InvoiceNo"]
    .astype("string")
    .str.startswith("C", na=False)
)

valid_purchase_mask = (
    (~cancelled_mask)
    & (df_prep["Quantity"] > 0)
    & (df_prep["UnitPrice"] > 0)
    & (df_prep["InvoiceDate"].notna())
)

sales = df_prep.loc[valid_purchase_mask].copy()

sales["LineTotal"] = sales["Quantity"] * sales["UnitPrice"]

print("Rows in prepared customer data:", len(df_prep))
print("Valid purchase rows:", len(sales))
print("Unique customers in valid purchases:", sales["CustomerID"].nunique())

print("\nDate range of valid purchases:")
print(sales["InvoiceDate"].min())
print(sales["InvoiceDate"].max())

print("\nValid purchase sample:")
display(sales.head())

Rows in prepared customer data: 401604
Valid purchase rows: 392692
Unique customers in valid purchases: 4338

Date range of valid purchases:
2010-12-01 08:26:00
2011-12-09 12:50:00

Valid purchase sample:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,LineTotal
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


## سؤال ۴: آیا بعد از آماده‌سازی اولیه، مشکلات مهمی در داده باقی مانده است؟

### پاسخ
پس از حذف ردیف‌های تکراری کامل و کنارگذاشتن ردیف‌های فاقد CustomerID، باید دوباره داده‌ها را بررسی کنیم.

این بررسی شامل موارد زیر است:

- تعداد مقادیر گمشده
- تعداد ردیف‌های تکراری
- مقادیر غیرمثبت Quantity و UnitPrice در مجموعه خرید معتبر
- تعداد مشتریان و فاکتورهای یکتا
- بازه زمانی تراکنش‌های معتبر

### چرا این سؤال را می‌پرسیم؟

برای اطمینان از اینکه تصمیم‌های آماده‌سازی به نتیجه مورد انتظار رسیده‌اند و مشکلاتی مانند تاریخ نامعتبر یا ردیف تکراری به‌صورت ناخواسته باقی نمانده‌اند.

### تصمیم آماده‌سازی

این بررسی یک کنترل کیفیت است. اگر مشکلی باقی مانده باشد، باید علت آن مشخص شود؛ نباید بدون بررسی، داده‌های بیشتری حذف شوند.

In [57]:
quality_report = pd.Series({
    "Rows in sales": len(sales),
    "Missing CustomerID": sales["CustomerID"].isna().sum(),
    "Missing InvoiceDate": sales["InvoiceDate"].isna().sum(),
    "Exact duplicate rows": sales.duplicated().sum(),
    "Non-positive Quantity": (sales["Quantity"] <= 0).sum(),
    "Non-positive UnitPrice": (sales["UnitPrice"] <= 0).sum(),
    "Unique customers": sales["CustomerID"].nunique(),
    "Unique invoices": sales["InvoiceNo"].nunique()
})

print(quality_report)

Rows in sales             392692
Missing CustomerID             0
Missing InvoiceDate            0
Exact duplicate rows           0
Non-positive Quantity          0
Non-positive UnitPrice         0
Unique customers            4338
Unique invoices            18532
dtype: int64


## سؤال ۵: چگونه ردیف‌های کالایی را به اطلاعات قابل استفاده برای تحلیل مشتری تبدیل کنیم؟

### پاسخ
دیتاست اولیه در سطح ردیف کالا است، اما پروژه ما در سطح مشتری انجام می‌شود. بنابراین باید از ردیف‌های خرید معتبر، شاخص‌هایی در سطح فاکتور و مشتری بسازیم.

برای هر ردیف خرید، مبلغ ردیف برابر است با:

LineTotal = Quantity × UnitPrice

سپس می‌توان اطلاعات را در سطح فاکتور و مشتری تجمیع کرد؛ برای مثال، مجموع مبلغ فاکتور، تعداد فاکتورهای مشتری و تعداد کالاهای خریداری‌شده.

### چرا این سؤال را می‌پرسیم؟

چون مدل پیش‌بینی ریزش باید رفتار مشتری را تحلیل کند، نه اینکه هر ردیف کالایی را یک مشتری یا یک خرید مستقل فرض کند.

### تصمیم آماده‌سازی

در این پروژه، مبلغ هر ردیف و سپس شاخص‌های تجمیعی مشتری محاسبه می‌شوند. تراکنش‌ها بر اساس CustomerID و بازه زمانی مربوط به هر تاریخ مرجع گروه‌بندی خواهند شد.

In [58]:
invoice_data = (
    sales.groupby(["CustomerID", "InvoiceNo"], as_index=False)
    .agg(
        InvoiceDate=("InvoiceDate", "max"),
        InvoiceAmount=("LineTotal", "sum"),
        TotalUnits=("Quantity", "sum"),
        ProductCount=("StockCode", "nunique")
    )
)

print("Invoice-level dataset shape:", invoice_data.shape)
display(invoice_data.head())

customer_summary = (
    invoice_data.groupby("CustomerID")
    .agg(
        InvoiceCount=("InvoiceNo", "nunique"),
        TotalSpend=("InvoiceAmount", "sum"),
        TotalUnits=("TotalUnits", "sum"),
        UniqueProducts=("ProductCount", "sum"),
        FirstPurchase=("InvoiceDate", "min"),
        LastPurchase=("InvoiceDate", "max")
    )
    .reset_index()
)

print("\nCustomer-level dataset shape:", customer_summary.shape)
display(customer_summary.head())

Invoice-level dataset shape: (18532, 6)


,CustomerID,InvoiceNo,InvoiceDate,InvoiceAmount,TotalUnits,ProductCount
0,12346,541431,2011-01-18 10:01:00,77183.60,74215,1
1,12347,537626,2010-12-07 14:57:00,711.79,319,31
2,12347,542237,2011-01-26 14:30:00,475.39,315,29
3,12347,549222,2011-04-07 10:43:00,636.25,483,24
4,12347,556201,2011-06-09 13:01:00,382.52,196,18



Customer-level dataset shape: (4338, 7)


,CustomerID,InvoiceCount,TotalSpend,TotalUnits,UniqueProducts,FirstPurchase,LastPurchase
0,12346,1,77183.60,74215,1,2011-01-18 10:01:00,2011-01-18 10:01:00
1,12347,7,4310.00,2458,182,2010-12-07 14:57:00,2011-12-07 15:52:00
2,12348,4,1797.24,2341,27,2010-12-16 19:09:00,2011-09-25 13:13:00
3,12349,1,1757.55,631,73,2011-11-21 09:51:00,2011-11-21 09:51:00
4,12350,1,334.40,197,17,2011-02-02 16:01:00,2011-02-02 16:01:00


## سؤال ۶: چه ویژگی‌هایی از رفتار گذشته مشتری برای پیش‌بینی ریزش ساخته می‌شوند؟

### پاسخ

ویژگی‌های اولیه پیشنهادی عبارت‌اند از:

- Recency: تعداد روزهای گذشته از آخرین خرید تا تاریخ مرجع
- Frequency: تعداد فاکتورهای خرید معتبر در بازه مشاهده
- Monetary: مجموع مبلغ خرید معتبر در بازه مشاهده
- TotalUnits: مجموع تعداد واحدهای خریداری‌شده
- UniqueProducts: تعداد محصولات متمایز خریداری‌شده
- ActiveDays: تعداد روزهای متمایزی که مشتری در آن‌ها خرید داشته است
- AverageInvoiceValue: میانگین مبلغ فاکتورهای مشتری
- AverageDaysBetweenPurchases: میانگین فاصله زمانی بین خریدهای متوالی، در صورت وجود حداقل دو فاکتور

### چرا این سؤال را می‌پرسیم؟

برای تبدیل داده‌های خام تراکنش به ویژگی‌هایی که الگوی خرید و تغییرات فعالیت مشتری را نشان می‌دهند. این ویژگی‌ها ورودی مدل پیش‌بینی خواهند بود.

### تصمیم آماده‌سازی

ویژگی‌ها باید فقط از تراکنش‌هایی ساخته شوند که قبل از تاریخ مرجع اتفاق افتاده‌اند. ویژگی‌هایی که از اطلاعات آینده استفاده کنند باعث نشت اطلاعات و ارزیابی غیرواقعی مدل می‌شوند.

In [59]:
def build_customer_features(sales_data, reference_date, history_days=180):
    reference_date = pd.Timestamp(reference_date).normalize()
    history_start = reference_date - pd.Timedelta(days=history_days)

    # Use purchases strictly before the reference date
    history = sales_data.loc[
        (sales_data["InvoiceDate"] >= history_start)
        & (sales_data["InvoiceDate"] < reference_date)
    ].copy()

    if history.empty:
        return pd.DataFrame()

    invoice_level = (
        history.groupby(["CustomerID", "InvoiceNo"], as_index=False)
        .agg(
            InvoiceDate=("InvoiceDate", "max"),
            InvoiceAmount=("LineTotal", "sum"),
            TotalUnits=("Quantity", "sum"),
            ProductCount=("StockCode", "nunique")
        )
    )

    features = (
        history.groupby("CustomerID")
        .agg(
            FirstPurchase=("InvoiceDate", "min"),
            LastPurchase=("InvoiceDate", "max"),
            TotalSpend=("LineTotal", "sum"),
            TotalUnits=("Quantity", "sum"),
            UniqueProducts=("StockCode", "nunique"),
            ActiveDays=("InvoiceDate", lambda x: x.dt.normalize().nunique())
        )
    )

    invoice_stats = (
        invoice_level.groupby("CustomerID")
        .agg(
            Frequency=("InvoiceNo", "nunique"),
            AverageInvoiceValue=("InvoiceAmount", "mean")
        )
    )

    features = features.join(invoice_stats, how="left")

    features["Recency"] = (
        reference_date - features["LastPurchase"].dt.normalize()
    ).dt.days

    features["TenureDays"] = (
        features["LastPurchase"].dt.normalize()
        - features["FirstPurchase"].dt.normalize()
    ).dt.days

    # Average gap between consecutive invoices
    invoice_level = invoice_level.sort_values(
        ["CustomerID", "InvoiceDate"]
    )

    invoice_level["DaysBetweenPurchases"] = (
        invoice_level.groupby("CustomerID")["InvoiceDate"]
        .diff()
        .dt.total_seconds()
        / 86400
    )

    average_gap = (
        invoice_level.groupby("CustomerID")["DaysBetweenPurchases"]
        .mean()
        .rename("AverageDaysBetweenPurchases")
    )

    features = features.join(average_gap, how="left")
    features["ReferenceDate"] = reference_date

    return features.reset_index()

In [60]:
reference_date = "2011-06-01"

customer_features = build_customer_features(
    sales_data=sales,
    reference_date=reference_date,
    history_days=180
)

print("Dataset shape:", customer_features.shape)

print("\nColumns:")
print(customer_features.columns.tolist())

display(customer_features.head())

Dataset shape: (2659, 13)

Columns:
['CustomerID', 'FirstPurchase', 'LastPurchase', 'TotalSpend', 'TotalUnits', 'UniqueProducts', 'ActiveDays', 'Frequency', 'AverageInvoiceValue', 'Recency', 'TenureDays', 'AverageDaysBetweenPurchases', 'ReferenceDate']


,CustomerID,FirstPurchase,LastPurchase,TotalSpend,TotalUnits,UniqueProducts,ActiveDays,Frequency,AverageInvoiceValue,Recency,TenureDays,AverageDaysBetweenPurchases,ReferenceDate
0,12346,2011-01-18 10:01:00,2011-01-18 10:01:00,77183.60,74215,1,1,1,77183.600000,134,0,NaN,2011-06-01
1,12347,2010-12-07 14:57:00,2011-04-07 10:43:00,1823.43,1117,63,3,3,607.810000,55,121,60.411806,2011-06-01
2,12348,2010-12-16 19:09:00,2011-04-05 10:47:00,1487.24,2124,22,3,3,495.746667,57,110,54.825694,2011-06-01
3,12350,2011-02-02 16:01:00,2011-02-02 16:01:00,334.40,197,17,1,1,334.400000,119,0,NaN,2011-06-01
4,12352,2011-02-16 12:33:00,2011-03-22 16:08:00,1561.81,254,26,4,5,312.362000,71,34,8.537326,2011-06-01


## سؤال ۷: برچسب Churn چگونه بر اساس رفتار آینده مشتری ساخته می‌شود؟

### پاسخ
دیتاست اصلی ستون آماده‌ای برای ریزش مشتری ندارد؛ بنابراین باید برچسب آن را تعریف کنیم.

تعریف عملیاتی این پروژه:

- پنجره مشاهده: 180 روز قبل از تاریخ مرجع
- پنجره تعیین برچسب: 90 روز از تاریخ مرجع به بعد
- Churn = 1: مشتری در 180 روز گذشته خرید معتبر داشته، اما در 90 روز آینده خرید معتبر ندارد.
- Churn = 0: مشتری در 90 روز آینده حداقل یک خرید معتبر انجام می‌دهد.

مشتریانی وارد این مجموعه می‌شوند که در پنجره مشاهده حداقل یک خرید معتبر داشته‌اند.

### چرا این سؤال را می‌پرسیم؟

مدل یادگیری نظارت‌شده برای آموزش به متغیر هدف نیاز دارد. تعریف روشن ریزش باعث می‌شود برچسب‌ها بر اساس یک قاعده یکسان ساخته شوند و نتایج مدل قابل تفسیر باشند.

### تصمیم آماده‌سازی

برای نمونه، تاریخ‌های مرجع 1 ژوئن، 1 ژوئیه، 1 اوت و 1 سپتامبر 2011 را بررسی می‌کنیم. برای هر تاریخ، ویژگی‌ها فقط از 180 روز قبل ساخته می‌شوند و برچسب از خریدهای 90 روز بعد به دست می‌آید.

این تعریف یک فرض عملیاتی برای پروژه است؛ مشتری بدون خرید در 90 روز آینده لزوماً برای همیشه از دست نرفته است.

In [61]:
reference_dates = [
    "2011-06-01",
    "2011-07-01",
    "2011-08-01",
    "2011-09-01"
]

snapshots = []

for ref_date in reference_dates:
    ref_date = pd.Timestamp(ref_date)

    features = build_customer_features(
        sales_data=sales,
        reference_date=ref_date,
        history_days=180
    )

    if features.empty:
        continue

    future_end = ref_date + pd.Timedelta(days=90)

    future_customers = sales.loc[
        (sales["InvoiceDate"] >= ref_date)
        & (sales["InvoiceDate"] < future_end),
        "CustomerID"
    ].unique()

    features["Churn"] = (
        ~features["CustomerID"].isin(future_customers)
    ).astype(int)

    snapshots.append(features)

if snapshots:
    churn_data = pd.concat(snapshots, ignore_index=True)
else:
    churn_data = pd.DataFrame()

print("Prepared dataset shape:", churn_data.shape)

if not churn_data.empty:
    print("\nRows per reference date:")
    print(churn_data["ReferenceDate"].value_counts().sort_index())

    print("\nChurn label distribution:")
    print(churn_data["Churn"].value_counts())

    print("\nChurn rate by reference date:")
    print(churn_data.groupby("ReferenceDate")["Churn"].mean())

    print("\nSample:")
    print(churn_data.head())

Prepared dataset shape: (10926, 14)

Rows per reference date:
ReferenceDate
2011-06-01    2659
2011-07-01    2724
2011-08-01    2771
2011-09-01    2772
Name: count, dtype: int64

Churn label distribution:
Churn
0    5970
1    4956
Name: count, dtype: int64

Churn rate by reference date:
ReferenceDate
2011-06-01    0.499436
2011-07-01    0.482746
2011-08-01    0.441718
2011-09-01    0.392857
Name: Churn, dtype: float64

Sample:
   CustomerID       FirstPurchase        LastPurchase  TotalSpend  TotalUnits  \
0       12346 2011-01-18 10:01:00 2011-01-18 10:01:00    77183.60       74215   
1       12347 2010-12-07 14:57:00 2011-04-07 10:43:00     1823.43        1117   
2       12348 2010-12-16 19:09:00 2011-04-05 10:47:00     1487.24        2124   
3       12350 2011-02-02 16:01:00 2011-02-02 16:01:00      334.40         197   
4       12352 2011-02-16 12:33:00 2011-03-22 16:08:00     1561.81         254   

   UniqueProducts  ActiveDays  Frequency  AverageInvoiceValue  Recency  \
0       

## سؤال ۸: آیا در ساخت ویژگی‌ها و برچسب ریزش، نشت اطلاعات آینده رخ داده است؟

### پاسخ
در این پروژه، ویژگی‌های مشتری فقط از تراکنش‌های 180 روز قبل از تاریخ مرجع ساخته می‌شوند. برچسب ریزش نیز فقط با بررسی خریدهای 90 روز بعد تعیین می‌شود.

ستون‌های `CustomerID` و `ReferenceDate` برای شناسایی مشتری و تاریخ هر نمونه لازم‌اند، اما نباید به‌عنوان ویژگی‌های عددی معمولی وارد مدل شوند.

### چرا این سؤال را می‌پرسیم؟

اگر مدل به اطلاعاتی دسترسی داشته باشد که در زمان پیش‌بینی هنوز در دسترس نیستند، عملکرد ارزیابی‌شده آن بیش از حد خوش‌بینانه می‌شود و در شرایط واقعی قابل اتکا نخواهد بود.

### تصمیم آماده‌سازی

ویژگی‌های رفتاری فقط از پنجره گذشته محاسبه می‌شوند. `Churn` متغیر هدف است و نباید در میان ویژگی‌های ورودی قرار بگیرد. شناسه مشتری نیز برای ردیابی نمونه‌ها حفظ می‌شود، اما به‌صورت مستقیم به مدل داده نمی‌شود.

In [62]:
if not churn_data.empty:

    excluded_from_features = [
        "CustomerID",
        "ReferenceDate",
        "FirstPurchase",
        "LastPurchase",
        "Churn"
    ]

    feature_columns = [
        column
        for column in churn_data.columns
        if column not in excluded_from_features
    ]

    X = churn_data[feature_columns].copy()
    y = churn_data["Churn"].copy()

    print("Feature columns:")
    print(feature_columns)

    print("\nTarget column:")
    print(y.name)

    print("\nFeatures contain target:", "Churn" in X.columns)
    print("Features contain CustomerID:", "CustomerID" in X.columns)
    print("Features contain ReferenceDate:", "ReferenceDate" in X.columns)

Feature columns:
['TotalSpend', 'TotalUnits', 'UniqueProducts', 'ActiveDays', 'Frequency', 'AverageInvoiceValue', 'Recency', 'TenureDays', 'AverageDaysBetweenPurchases']

Target column:
Churn

Features contain target: False
Features contain CustomerID: False
Features contain ReferenceDate: False


In [63]:
print("Feature columns:")
print(feature_columns)

print("\nchurn_data:")
display(churn_data)

suspicious_columns = [
    "Churn",
    "CustomerID",
    "ReferenceDate",
    "FirstPurchase",
    "LastPurchase"
]

print("\nPotential leakage columns in features:")
print([col for col in suspicious_columns if col in feature_columns])

Feature columns:
['TotalSpend', 'TotalUnits', 'UniqueProducts', 'ActiveDays', 'Frequency', 'AverageInvoiceValue', 'Recency', 'TenureDays', 'AverageDaysBetweenPurchases']

churn_data:


,CustomerID,FirstPurchase,LastPurchase,TotalSpend,TotalUnits,UniqueProducts,ActiveDays,Frequency,AverageInvoiceValue,Recency,TenureDays,AverageDaysBetweenPurchases,ReferenceDate,Churn
0,12346,2011-01-18 10:01:00,2011-01-18 10:01:00,77183.60,74215,1,1,1,77183.600000,134,0,NaN,2011-06-01,1
1,12347,2010-12-07 14:57:00,2011-04-07 10:43:00,1823.43,1117,63,3,3,607.810000,55,121,60.411806,2011-06-01,0
2,12348,2010-12-16 19:09:00,2011-04-05 10:47:00,1487.24,2124,22,3,3,495.746667,57,110,54.825694,2011-06-01,1
3,12350,2011-02-02 16:01:00,2011-02-02 16:01:00,334.40,197,17,1,1,334.400000,119,0,NaN,2011-06-01,1
4,12352,2011-02-16 12:33:00,2011-03-22 16:08:00,1561.81,254,26,4,5,312.362000,71,34,8.537326,2011-06-01,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10921,18280,2011-03-07 09:52:00,2011-03-07 09:52:00,180.60,45,10,1,1,180.600000,178,0,NaN,2011-09-01,1
10922,18281,2011-06-12 10:53:00,2011-06-12 10:53:00,80.82,54,7,1,1,80.820000,81,0,NaN,2011-09-01,1
10923,18282,2011-08-05 13:35:00,2011-08-05 13:35:00,100.21,75,7,1,1,100.210000,27,0,NaN,2011-09-01,1
10924,18283,2011-04-21 16:37:00,2011-07-14 13:20:00,637.23,439,124,5,6,106.205000,49,84,16.772639,2011-09-01,0



Potential leakage columns in features:
[]


## سؤال ۹: آیا داده‌های آماده‌شده برای ورود به مرحله Modeling مناسب‌اند؟

### پاسخ
برای ورود به مرحله مدل‌سازی باید بررسی کنیم:

- مجموعه ویژگی‌ها و برچسب ریزش ساخته شده باشند.
- ستون هدف فقط شامل مقادیر 0 و 1 باشد.
- مقادیر گمشده ویژگی‌ها شناسایی شده باشند.
- ویژگی‌ها فقط از اطلاعات مجاز قبل از تاریخ مرجع ساخته شده باشند.
- توزیع برچسب ریزش و تعداد نمونه‌های هر تاریخ مرجع بررسی شده باشد.
- شناسه‌ها و ستون‌های کمکی از ورودی مدل کنار گذاشته شده باشند.

### چرا این سؤال را می‌پرسیم؟

برای اطمینان از اینکه خروجی Data Preparation از نظر ساختار و منطق هدف قابل استفاده است و مشکلات مهم به مرحله مدل‌سازی منتقل نمی‌شوند.

### تصمیم آماده‌سازی

در این مرحله داده‌ها را ارزیابی می‌کنیم، اما هنوز مدل آموزش نمی‌دهیم. اگر ویژگی‌ها دارای مقادیر گمشده باشند، روش مدیریت آن‌ها باید با توجه به نوع ویژگی و الگوریتم انتخاب شود و فقط روی داده آموزش برازش شود.

In [64]:
if churn_data.empty:
    print("No prepared snapshots were created.")
else:
    print("Dataset shape:", churn_data.shape)

    print("\nMissing values in each column:")
    print(churn_data.isna().sum())

    print("\nTarget values:")
    print(sorted(churn_data["Churn"].dropna().unique()))

    print("\nTarget distribution:")
    print(churn_data["Churn"].value_counts(dropna=False))

    print("\nTarget rate by reference date:")
    print(
        churn_data.groupby("ReferenceDate")["Churn"]
        .agg(["count", "mean"])
    )

    print("\nDuplicate customer-reference pairs:")
    print(
        churn_data.duplicated(
            subset=["CustomerID", "ReferenceDate"]
        ).sum()
    )

Dataset shape: (10926, 14)

Missing values in each column:
CustomerID                        0
FirstPurchase                     0
LastPurchase                      0
TotalSpend                        0
TotalUnits                        0
UniqueProducts                    0
ActiveDays                        0
Frequency                         0
AverageInvoiceValue               0
Recency                           0
TenureDays                        0
AverageDaysBetweenPurchases    5166
ReferenceDate                     0
Churn                             0
dtype: int64

Target values:
[np.int64(0), np.int64(1)]

Target distribution:
Churn
0    5970
1    4956
Name: count, dtype: int64

Target rate by reference date:
               count      mean
ReferenceDate                 
2011-06-01      2659  0.499436
2011-07-01      2724  0.482746
2011-08-01      2771  0.441718
2011-09-01      2772  0.392857

Duplicate customer-reference pairs:
0


## سؤال ۱۰: خروجی نهایی Data Preparation در چه قالبی ذخیره می‌شود؟

### پاسخ
خروجی نهایی شامل یک ردیف برای هر مشتری در هر تاریخ مرجع است. این جدول، ویژگی‌های محاسبه‌شده از پنجره مشاهده و ستون هدف `Churn` را در بر می‌گیرد.

### چرا این سؤال را می‌پرسیم؟

برای اینکه مرحله Modeling بتواند داده آماده‌شده را به‌صورت منظم دریافت کند و نتایج آماده‌سازی قابل بازتولید باشند.

### تصمیم آماده‌سازی

مجموعه نهایی را در یک فایل CSV جداگانه ذخیره می‌کنیم. داده خام را بازنویسی نمی‌کنیم. پیش از استفاده در مدل، باید طرح ارزیابی زمانی و نحوه تقسیم داده‌ها نیز مشخص شود.

In [65]:
if not churn_data.empty:
    output_path = "Customer_Churn_Prepared.csv"

    churn_data.to_csv(
        output_path,
        index=False
    )

    print("Prepared dataset saved to:", output_path)
    print("Final shape:", churn_data.shape)
else:
    print("Nothing to save: churn_data is empty.")

Prepared dataset saved to: Customer_Churn_Prepared.csv
Final shape: (10926, 14)
